In [1]:
import os

if os.path.exists("NovaS.pdf"):
    print("PDF is exists")

PDF is exists


In [2]:

from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from dotenv import load_dotenv
load_dotenv()

/Users/rahulyadav/Project/AI_fundamentals/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


True

## STEP-0 : Load the pdf into text format

In [3]:
text_data= PyPDFLoader("NovaS.pdf").load()

for page in text_data:
    page.metadata["source"]="Novas.pdf"
    page.metadata["author"]="Rahul Yadav"

#text_data

## STEP-1: creating Chunks of the data

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks=splitter.split_documents(text_data)
len(chunks)

101

## STEP-2: Creating Embeddigs of the chunks

In [5]:
from langchain_openai import OpenAIEmbeddings

embed_model=OpenAIEmbeddings(model="text-embedding-3-small")

embedded_chunks=embed_model.embed_documents([i.page_content for i in chunks])
len(embedded_chunks)

101

## STEP-2&3 : Creating Embedding and Storing them in Vector DB

In [6]:
from langchain_community.vectorstores import Chroma

embed_model=OpenAIEmbeddings(model="text-embedding-3-small")

chroma_db=Chroma.from_documents(chunks,embed_model,persist_directory="./chroma_db")

## STEP-4: Conection & Retrieval 

In [8]:
chroma_db_con=Chroma(persist_directory="./chroma_db", embedding_function=embed_model)

/var/folders/cr/34qx8njd48g07ps00m4fshq40000gn/T/ipykernel_21428/2423763980.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con=Chroma(persist_directory="./chroma_db", embedding_function=embed_model)


In [10]:
chroma_db_con.similarity_search("When NovaSphere Technologies was founded", k=3)

[Document(metadata={'creationdate': '2026-03-31T11:24:15-03:00', 'page_label': '1', 'moddate': '2026-03-31T11:24:15-03:00', 'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'total_pages': 3, 'source': 'Novas.pdf', 'author': 'Rahul Yadav', 'page': 0}, page_content='In 2018, NovaSphere Technologies started getting more attention in the local technology'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'total_pages': 3, 'page': 2, 'source': 'Novas.pdf', 'moddate': '2026-03-31T11:24:15-03:00', 'creationdate': '2026-03-31T11:24:15-03:00', 'page_label': '3', 'author': 'Rahul Yadav', 'creator': 'Microsoft® Word for Microsoft 365'}, page_content='Today, NovaSphere Technologies is considered a reliable organization that provides data'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'source': 'Novas.pdf', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'moddat